In [ ]:
# === Setup ===
# Runtime: <1m with OAI_FAST_MODE=1
# Hardware: CPU smoke
# Network: none
# Competition-safe: Yes for the declared profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# 3. Experiments: L1/L2 Regularization

Khi dữ liệu có quá nhiều features hoặc có outlier, Linear Regression dễ bị Overfitting. Để giải quyết, người ta dùng **Regularization** bằng cách cộng thêm một thành phần phạt (penalty) vào hàm Loss.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Tạo dữ liệu phức tạp (phi tuyến tính) nhưng ta cố tình fit đa thức bậc cao để gây Overfitting
np.random.seed(42)
X = 2 * np.random.rand(20, 1)
y = 0.5 * X**2 + X + 2 + np.random.randn(20, 1)

# Tạo pipeline: Tạo đa thức bậc 10 (chắc chắn overfit với 20 data points)
X_plot = np.linspace(0, 2, 100).reshape(-1, 1)

models = {
    'Linear (Overfit)': LinearRegression(),
    'Ridge (L2)': Ridge(alpha=1.0),
    'Lasso (L1)': Lasso(alpha=0.1)
}

plt.figure(figsize=(15, 5))

for i, (name, model) in enumerate(models.items()):
    poly_model = make_pipeline(PolynomialFeatures(10), model)
    poly_model.fit(X, y)
    
    y_plot = poly_model.predict(X_plot)
    
    plt.subplot(1, 3, i+1)
    plt.scatter(X, y, color='blue')
    plt.plot(X_plot, y_plot, color='red')
    plt.title(name)
    plt.ylim(0, 8)
    
    # In trọng số để thấy L1 làm 0 nhiều trọng số như thế nào
    coefs = poly_model.steps[1][1].coef_.flatten()
    non_zero = np.sum(np.abs(coefs) > 1e-5)
    print(f'{name} - Số tham số khác không: {non_zero}')

plt.tight_layout()
plt.show()

### Nhận xét (Hypothesis vs Observation)
1. **LinearRegression** với PolynomialFeatures(10) sẽ cực kỳ gấp khúc, fit vào nhiễu (Overfitting).
2. **Ridge (L2 Regularization)** sẽ kéo các trọng số về gần 0, làm mượt đường cong.
3. **Lasso (L1 Regularization)** không chỉ kéo về gần 0 mà còn ép rất nhiều trọng số bằng chính xác 0 (Feature selection).